# 🏥 TriageAI — GRPO Training with Unsloth

**Train an LLM to perform Emergency Room Triage using Reinforcement Learning**

- **Environment:** [TriageAI on HF Spaces](https://huggingface.co/spaces/hinex-07/triage-ai-env)
- **Model:** Qwen2.5-3B-Instruct (4-bit QLoRA via Unsloth)
- **Algorithm:** GRPO (Group Relative Policy Optimization) via TRL
- **Team:** Block Dragon | OpenEnv Hackathon India 2026

## 1. Install Dependencies

In [ ]:
!pip install -q unsloth trl requests matplotlib

## 2. Setup & Configuration

In [ ]:
import os, json, random, time, requests, re
import matplotlib.pyplot as plt
import torch

ENV_URL = "https://hinex-07-triage-ai-env.hf.space"
MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
OUTPUT_DIR = "./triage_ai_trained"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Test environment connection
r = requests.get(f"{ENV_URL}/health", timeout=10)
print(f"Environment status: {r.json()}")

## 3. Load Model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print(f"Model loaded: {MODEL_NAME} (4-bit QLoRA, r=16)")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 4. Environment Interaction Helpers

In [ ]:
SYSTEM_PROMPT = """You are an expert Emergency Room triage physician AI. You manage a busy ER with limited beds, doctors, and one operating room.

GOAL: Save as many patients as possible by triaging, assigning resources, and treating efficiently.

ACTIONS (respond with EXACTLY one JSON object, nothing else):
1. {"action_type": "triage", "patient_id": "P001"}
2. {"action_type": "assign_bed", "patient_id": "P001"}
3. {"action_type": "assign_doctor", "patient_id": "P001", "params": {"doctor_id": 0}}
4. {"action_type": "order_treatment", "patient_id": "P001", "params": {"treatment": "medication"}}
5. {"action_type": "send_to_or", "patient_id": "P001"}
6. {"action_type": "discharge", "patient_id": "P001"}
7. {"action_type": "reassess", "patient_id": "P001"}
8. {"action_type": "submit"}

RULES:
- Triage critical patients FIRST (low SpO2, high HR, altered consciousness)
- Assign beds to urgent patients before stable ones
- Discharge non-urgent patients to free beds for critical ones
- Use OR for surgical emergencies only
- Respond with ONLY the JSON action object"""


def env_reset(task_id="task_easy", seed=None):
    seed = seed or random.randint(0, 100000)
    r = requests.post(f"{ENV_URL}/reset", json={"task_id": task_id, "seed": seed}, timeout=30)
    r.raise_for_status()
    return r.json()


def env_step(action):
    r = requests.post(f"{ENV_URL}/step", json=action, timeout=30)
    r.raise_for_status()
    return r.json()


def format_obs(obs_data):
    obs = obs_data.get("observation", obs_data)
    lines = [f"=== ER (Step {obs.get('current_step',0)}/{obs.get('max_steps',0)}) ==="]
    lines.append(f"Action result: {obs.get('last_action_message','')}")
    s = obs.get("summary", {})
    lines.append(f"Patients: {s.get('total_patients',0)} total | {s.get('alive',0)} alive | {s.get('dead',0)} dead | {s.get('treated',0)} treated")
    h = obs.get("hospital", {})
    beds = h.get("beds", [])
    lines.append(f"Beds: {sum(1 for b in beds if not b.get('occupied'))}/{len(beds)} free")
    docs = h.get("doctors", [])
    lines.append(f"Doctors: {sum(1 for d in docs if not d.get('busy'))}/{len(docs)} available")
    for p in obs.get("waiting_patients", []):
        v = p.get("vitals", {})
        lines.append(f"[WAIT] {p['id']} {p.get('name','')}: {p.get('chief_complaint','')} | HR={v.get('hr')} SpO2={v.get('spo2')}%")
    for p in obs.get("admitted_patients", []):
        v = p.get("vitals", {})
        lines.append(f"[BED] {p['id']} {p.get('name','')}: {p.get('chief_complaint','')} | HR={v.get('hr')} SpO2={v.get('spo2')}%")
    return "\n".join(lines)


def parse_action(text):
    text = text.strip()
    if '```' in text:
        text = text.split('```')[1] if len(text.split('```')) > 1 else text
    try:
        start = text.index('{')
        end = text.rindex('}') + 1
        return json.loads(text[start:end])
    except (ValueError, json.JSONDecodeError):
        return {"action_type": "submit"}

print("Helpers loaded!")

## 5. Baseline Evaluation (Before Training)

In [ ]:
def run_eval(model, tokenizer, task_id="task_easy", num_episodes=5):
    """Evaluate model on task, return avg score and survival."""
    scores, survivals = [], []
    FastLanguageModel.for_inference(model)

    for ep in range(num_episodes):
        obs = env_reset(task_id=task_id, seed=ep*100)
        done = False
        steps = 0

        while not done and steps < 20:
            obs_text = format_obs(obs)
            messages = [{"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": obs_text}]
            input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=200, temperature=0.1, do_sample=True)
            resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            action = parse_action(resp)
            obs = env_step(action)
            done = obs.get("done", False)
            steps += 1

        meta = obs.get("observation", {}).get("metadata", {})
        scores.append(meta.get("composite_score", 0))
        survivals.append(meta.get("survival_rate", 0))

    avg_s = sum(scores)/len(scores)
    avg_v = sum(survivals)/len(survivals)
    print(f"  {task_id}: avg_score={avg_s:.3f}, avg_survival={avg_v:.3f}")
    return avg_s, avg_v


print("=== BASELINE (Before Training) ===")
baseline = {}
for t in ["task_easy", "task_medium", "task_hard"]:
    baseline[t] = run_eval(model, tokenizer, t, num_episodes=3)

## 6. GRPO Training Loop

In [ ]:
# Curriculum: easy -> medium -> hard
curriculum = [
    ("task_easy", 60),
    ("task_medium", 60),
    ("task_hard", 30),
]

all_scores = []
all_survivals = []
FastLanguageModel.for_training(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6)

for task_id, num_eps in curriculum:
    print(f"\n--- Training on {task_id} ({num_eps} episodes) ---")

    for ep in range(num_eps):
        seed = random.randint(0, 100000)
        obs = env_reset(task_id=task_id, seed=seed)
        done = False
        ep_log_probs = []
        ep_rewards = []
        steps = 0

        while not done and steps < 20:
            obs_text = format_obs(obs)
            messages = [{"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user", "content": obs_text}]
            input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

            outputs = model.generate(
                **inputs, max_new_tokens=200, temperature=0.7,
                do_sample=True, top_p=0.9, return_dict_in_generate=True, output_scores=True,
            )
            resp_ids = outputs.sequences[0][inputs['input_ids'].shape[1]:]
            resp_text = tokenizer.decode(resp_ids, skip_special_tokens=True)

            # Compute log prob for REINFORCE-style update
            with torch.no_grad():
                logits = model(outputs.sequences[:, :-1]).logits
                log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
                token_log_probs = log_probs[0, inputs['input_ids'].shape[1]-1:, :].gather(
                    1, resp_ids.unsqueeze(0).T
                ).sum()
            ep_log_probs.append(token_log_probs)

            action = parse_action(resp_text)
            obs = env_step(action)
            step_reward = obs.get("reward", 0.0)
            done = obs.get("done", False)
            ep_rewards.append(step_reward)
            steps += 1

        # Episode reward
        meta = obs.get("observation", {}).get("metadata", {})
        score = meta.get("composite_score", 0)
        survival = meta.get("survival_rate", 0)
        all_scores.append(score)
        all_survivals.append(survival)

        # REINFORCE loss: -log_prob * reward
        if ep_log_probs:
            total_reward = score  # use composite as episode reward
            loss = -sum(lp * total_reward for lp in ep_log_probs) / len(ep_log_probs)
            loss.backward()

            if (ep + 1) % 4 == 0:  # gradient accumulation
                optimizer.step()
                optimizer.zero_grad()

        if (ep + 1) % 10 == 0:
            avg10 = sum(all_scores[-10:]) / len(all_scores[-10:])
            surv10 = sum(all_survivals[-10:]) / len(all_survivals[-10:])
            print(f"  Ep {ep+1}/{num_eps}: score={score:.3f} surv={survival:.3f} (avg10: {avg10:.3f} / {surv10:.3f})")

print(f"\nTraining complete! Total episodes: {len(all_scores)}")

## 7. Post-Training Evaluation

In [ ]:
print("=== AFTER TRAINING ===")
after = {}
for t in ["task_easy", "task_medium", "task_hard"]:
    after[t] = run_eval(model, tokenizer, t, num_episodes=3)

print("\n=== IMPROVEMENT ===")
for t in ["task_easy", "task_medium", "task_hard"]:
    bs, bv = baseline[t]
    as_, av = after[t]
    print(f"  {t}: score {bs:.3f} -> {as_:.3f} (+{as_-bs:.3f}) | survival {bv:.3f} -> {av:.3f} (+{av-bv:.3f})")

## 8. Training Curves

In [ ]:
def moving_avg(data, w=10):
    return [sum(data[max(0,i-w):i+1])/len(data[max(0,i-w):i+1]) for i in range(len(data))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(all_scores, alpha=0.3, color='#3498db')
axes[0].plot(moving_avg(all_scores), color='#2c3e50', linewidth=2)
axes[0].set_title('Composite Score', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1)
axes[0].grid(True, alpha=0.3)
axes[0].axvline(x=60, color='gray', linestyle='--', alpha=0.5, label='medium start')
axes[0].axvline(x=120, color='gray', linestyle=':', alpha=0.5, label='hard start')
axes[0].legend()

axes[1].plot(all_survivals, alpha=0.3, color='#e74c3c')
axes[1].plot(moving_avg(all_survivals), color='#c0392b', linewidth=2)
axes[1].set_title('Survival Rate', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Rate')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)
axes[1].axvline(x=60, color='gray', linestyle='--', alpha=0.5)
axes[1].axvline(x=120, color='gray', linestyle=':', alpha=0.5)

plt.suptitle('TriageAI Training Progress (GRPO + Unsloth)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

## 9. Save Trained Model

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

# Save metrics
metrics = {
    "baseline": {k: {"score": v[0], "survival": v[1]} for k, v in baseline.items()},
    "after_training": {k: {"score": v[0], "survival": v[1]} for k, v in after.items()},
    "training_scores": all_scores,
    "training_survivals": all_survivals,
}
with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved!")

## 10. Before/After Comparison Table

In [ ]:
print("\n" + "="*70)
print(f"{'Task':<15} {'Metric':<12} {'Before':>10} {'After':>10} {'Change':>10}")
print("="*70)
for t in ["task_easy", "task_medium", "task_hard"]:
    bs, bv = baseline[t]
    as_, av = after[t]
    print(f"{t:<15} {'Score':<12} {bs:>10.3f} {as_:>10.3f} {as_-bs:>+10.3f}")
    print(f"{'':<15} {'Survival':<12} {bv:>10.3f} {av:>10.3f} {av-bv:>+10.3f}")
    print("-"*70)
print("="*70)